# 2. Readers, binners, and spatial coordinates

An image context combines three independent components: a reader for sparse spectra, a forward binner for a regular model input grid, and an optional inverse binner for converting decoded grids back to sparse `(m/z, intensity)` pairs.

In [1]:
import os 
from pathlib import Path

# seting global dir
cwd=Path.cwd()
if cwd.name == "tutorials":
    # os.chdir(cwd.parent.parent) 
    os.chdir(cwd.parent.parent.parent) 
os.getcwd()

'/home/maxi7524/repositories/MSIAutoEncoderWrapper'

In [2]:
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

wrapper = MSIAutoEncoderWrapper(
    project_path="data/tutorial_workspace",
    coordinate_order="xy",
)
image_path = Path("data/tutorial_workspace/imgs/example.imzML").resolve()
assert image_path.is_file(), "Download/copy the example imzML and ibd pair first."

2026-07-19 15:37:22,678 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets'.
2026-07-19 15:37:22,680 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 14 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders'.
2026-07-19 15:37:22,683 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 0 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.schema'.
2026-07-19 15:37:22,684 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 20 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.
2026-07-19 15:37:22,695 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 6 implementation module(s) in package 'msi_autoencoder_wrapper.training.criterions.autoencoder'.
2026-07-19 15

In [3]:
wrapper.workspace.set_default_image_path('example')

2026-07-19 15:37:27,038 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:93 | Default fallback image anchored: example


## Discover and configure components

The discovery methods print registered names and constructor parameters in one consistent format. Registered names plus JSON-compatible parameters are preferred: they can be serialized and reconstructed when a model is loaded.

In [5]:
wrapper.context_manager.get_available_inverse_binners()


 REGISTERED MSI INVERSE BINNERS & PARAMETERS

[Inverse Binner Key]: 'TopPeaksInverseBinner'
 Description: Resolution reduction algorithm tracking top peak heights across non-overlapping contextual coordinate masks.
 Parameters (kwargs):
   - binner: None
   - max_bins: 500
   - window_size: 3
   - active_context: None



In [7]:
wrapper.context_manager.get_available_readers()
wrapper.context_manager.get_available_binners()
wrapper.context_manager.get_available_inverse_binners()

# Remark - set_reader by uses set_active_context, when default value is provided it will use ut ('example.imzML' in that case)
reader = wrapper.context_manager.set_reader("PyImzMLReader")
reader = wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
binner = wrapper.context_manager.set_binner(
    "LinearBinning",
    str(image_path),
    # REMARK: If you are using m2aia reader you can obtain this value by: #TODO 
    bin_step=0.1,
)
inverse = wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner",
    str(image_path),
    max_bins=1500,
    window_size=3,
)


 REGISTERED MSI READERS & PARAMETERS

[Reader Key]: 'M2aiaReader'
 Description: Concrete data loader adapter linking native binary C++ M2aia bindings to the library ecosystem.
 Parameters (kwargs):
   - file_path: Required
   - active_context: None

[Reader Key]: 'PyImzMLReader'
 Description: Read imzML/ibd pairs through pyimzML without requiring M²aia.
 Parameters (kwargs):
   - file_path: Required
   - active_context: None


 REGISTERED MSI FORWARD BINNERS & PARAMETERS

[Binner Key]: 'LinearBinning'
 Description: Concrete processing strategy executing fast linear quantization via equidistant mass-to-charge bins.
 Parameters (kwargs):
   - bin_step: Required
   - x_min: None
   - x_max: None
   - active_context: None


 REGISTERED MSI INVERSE BINNERS & PARAMETERS

[Inverse Binner Key]: 'TopPeaksInverseBinner'
 Description: Resolution reduction algorithm tracking top peak heights across non-overlapping contextual coordinate masks.
 Parameters (kwargs):
   - binner: None
   - max_bins:

`PyImzMLReader` and `M2aiaReader` implement the same wrapper reader contract, so either can back datasets and training. M²aia uses native code and is the preferred performance-oriented backend where its binaries are supported. The pure-Python pyimzML backend is the portability fallback, including macOS installations where M²aia is unavailable. Treat performance as dataset- and installation-dependent and benchmark the intended workflow.

Ready instances and classes are also accepted. This is useful during development, but the constructor still has to expose a complete serializable config if the context is later saved.

In [ ]:
from msi_autoencoder_wrapper.readers.strategies.pyimzml_reader import PyImzMLReader
from msi_autoencoder_wrapper.binners.binners_strategies.linear_binner import LinearBinning


#TODO - zunifkować ten fragment, ponieawz nie jest to kompatybiolne żeby puścić dwie takei runy bez crasha (ttylko tą gónrą kormę tzeba uruchomić, powinno byc tak że jest tutorail wersja z m2aia i z tą drugą


# Case: active image
## Remark - here we use `set_active_image` to provide context 
wrapper.workspace.set_active_image(str(image_path)) 

## setup 
custom_reader = PyImzMLReader(image_path, active_context=wrapper.active_context)
## This replaces the reader stored for the same image context.
wrapper.context_manager.set_reader(custom_reader, str(image_path))
custom_binner = LinearBinning(bin_step=0.1, active_context=wrapper.active_context)
wrapper.context_manager.set_binner(custom_binner, str(image_path))
wrapper.workspace.clear_active_context()

# Case: default image

## setup 
custom_reader = PyImzMLReader(image_path, active_context=wrapper.active_context)
## This replaces the reader stored for the same image context.
wrapper.context_manager.set_reader(custom_reader, str(image_path))
custom_binner = LinearBinning(bin_step=0.1, active_context=wrapper.active_context)
wrapper.context_manager.set_binner(custom_binner, str(image_path))
 

2026-07-19 14:34:53,749 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:69 | Active image set via direct filesystem path: example (Location: /home/maxi7524/repositories/MSIAutoEncoderWrapper/data/tutorial_workspace/imgs)
2026-07-19 14:34:56,090 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:69 | Active image set via direct filesystem path: example (Location: /home/maxi7524/repositories/MSIAutoEncoderWrapper/data/tutorial_workspace/imgs)
2026-07-19 14:34:56,091 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:323 | Resolving system component 'reader' under image context 'example'
2026-07-19 14:34:56,091 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:338 | Successfully registered component 'reader' into ledger for image 'example'
2026-07-19 14:34:56,092 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.activ

## Raw, binned, and inverse-binned spectra

Readers return the original sparse axis and intensities. A binner sums signal into a fixed grid required by a neural network. An inverse binner is lossy: it selects a bounded set of important grid bins; it cannot restore information discarded during binning.

In [8]:
wrapper.active_context.reader[0:2, 5:8]

2026-07-19 15:39:21,718 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:68 | No active image set. Automatically resolving default workspace context: example
2026-07-19 15:39:21,720 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:53 | No image name provided. Falling back to default workspace image configuration.
2026-07-19 15:39:21,721 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:108 | Successfully bound active context memory maps for: example


{(1,
  5,
  1): (array([403.02190115, 403.02495771, 403.02801429, ..., 997.58327588,
         997.59517915, 997.60708263], shape=(2862,)), array([0., 0., 0., ..., 0., 0., 0.], shape=(2862,), dtype=float32)),
 (1,
  6,
  1): (array([404.03203959, 404.03510765, 404.03817574, ..., 994.16333351,
         994.17517563, 994.18701796], shape=(1734,)), array([0., 0., 0., ..., 0., 0., 0.], shape=(1734,), dtype=float32)),
 (1,
  7,
  1): (array([403.82084113, 403.82390678, 403.82697246, ..., 988.19889968,
         988.21063538, 988.22237129], shape=(1648,)), array([0., 0., 0., ..., 0., 0., 0.], shape=(1648,), dtype=float32))}

In [9]:
xs, ys = wrapper.active_context.reader[0]
xs, ys 

(array([404.09271947, 404.09578822, 404.09885701, ..., 988.2089146 ,
        988.2206505 , 988.23238661], shape=(1129,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(1129,), dtype=float32))

In [13]:
grid_ys = wrapper.active_context.binner(xs=xs, ys=ys)
grid_ys

array([1196.96820068,    0.        ,    0.        , ...,    0.        ,
       1509.52175903,    0.        ], shape=(5842,))

In [14]:
restored_xs, restored_ys = wrapper.active_context.inverse_binner(grid_ys)
restored_xs, restored_ys

(array([404.14271947, 404.24271947, 404.34271947, ..., 988.04271947,
        988.14271947, 988.24271947], shape=(1500,)),
 array([1196.96820068,    0.        ,    0.        , ...,    0.        ,
        1509.52175903,    0.        ], shape=(1500,)))

## Indexing, coordinates, and slices

An integer selects a flat spectrum index. A coordinate tuple selects one pixel. A normal Python slice selects flat indices, while a tuple containing slices selects coordinate values. Slice bounds are coordinate values, not zero-based matrix offsets.

In [16]:
reader = wrapper.active_context.reader
first_spectrum = reader[0]
first_ten = reader[:10]
x, y, z = reader.GetSpectrumPosition(0)
same_spectrum = reader[(x, y, z)]
region = reader[(slice(x, x + 5), slice(y, y + 5), z)]
print(len(first_ten), len(region))

10 25


`coordinate_order="xy"` exposes `(x, y, z)`. Matrix-oriented code often reads more naturally as `(row, column, z)`, which reverses the first two stored axes. The setting is wrapper-wide and affects original and latent readers consistently; it can be changed at runtime.

In [17]:
wrapper.set_coordinate_order("matrix")
row, column = y, x
matrix_spectrum = reader[(row, column, z)]
matrix_region = wrapper.active_context.get_region(
    slice(row, row + 5),
    slice(column, column + 5),
    z,
    source="image",
)
matrix_region

{(1,
  1,
  1): (array([404.09271947, 404.09578822, 404.09885701, ..., 988.2089146 ,
         988.2206505 , 988.23238661], shape=(1129,)), array([0., 0., 0., ..., 0., 0., 0.], shape=(1129,), dtype=float32)),
 (1,
  2,
  1): (array([409.92429493, 409.92743035, 409.93056581, 409.9337013 ,
         409.93683683, 409.9399724 , 409.943108  , 409.94624364,
         409.94937931, 409.95249394, 409.95562968, 409.95876546,
         409.96190128, 418.93390699, 418.93714634, 418.94038574,
         418.94362517, 418.94686464, 418.95010415, 418.95334369,
         418.95658328, 418.9598229 , 418.96306255, 418.96626489,
         418.96950463, 418.9727444 , 418.9759842 , 439.46422179,
         439.46770216, 439.47118258, 439.47466304, 439.47814354,
         439.48162408, 439.48510466, 439.48858528, 439.49206595,
         439.49554459, 439.49902534, 439.50250612, 439.50598695,
         440.83170185, 440.83519848, 440.83869516, 440.84219187,
         440.84568863, 440.84918543, 440.85268227, 440.8561791

## Presets and reproducible customization

Reader and binner setup is stored as component names plus parameters. Architecture presets use the same principle and are covered in Tutorial 3. Custom registered components and presets should retain every constructor parameter in their config; see [Custom models](../../../docs/CUSTOM_MODELS.md).

> **Important considerations — TODO:** add domain-specific guidance for profile/centroid data, normalization choices, calibration, and future reader methods.